# Build Autonomous Agent Prediction submission

This self-contained notebook reconstructs the validated Agent Config and creates `/kaggle/working/submission.zip`. No internet or dataset attachment is required.

In [ ]:
from pathlib import Path
import base64, json, shutil, zipfile

FILES = json.loads("{\"agent.yaml\": \"bmFtZTogaW50ZXJhY3Rpb25fcm91dGVkX2F1dG9tbF92NQpkZXNjcmlwdGlvbjogRmFpbC1zYWZlIGFkYXB0aXZlIEF1dG9NTCB3aXRoIHJvdXRlZCBxdWFkcmF0aWMgaW50ZXJhY3Rpb25zLCBwcmVzZXJ2ZWQgYmFzZWxpbmVzLCBhbmQgZXZpZGVuY2UtYmFzZWQgc2VsZWN0aW9uLgptb2RlbDogZ2VtaW5pLTMuNS1mbGFzaAppbnN0cnVjdGlvbjogIWluY2x1ZGUgcHJvbXB0cy9zeXN0ZW0ubWQKdG9vbHM6CiAgLSBydW5fY29tbWFuZAogIC0gc3VibWl0X3ByZWRpY3Rpb25zCiAgLSBzZWxlY3Rfc3VibWlzc2lvbgogIC0gZ2V0X3N0YXR1cwpza2lsbHM6CiAgLSBza2lsbHMvdGFidWxhci1hdXRvbWwKZ2VuZXJhdGVfY29udGVudF9jb25maWc6ICFpbmNsdWRlIGNvbmZpZ3Mvc2FtcGxpbmcueWFtbAo=\", \"configs/sampling.yaml\": \"dGVtcGVyYXR1cmU6IDAuMQptYXhfb3V0cHV0X3Rva2VuczogNDA5Ngp0aGlua2luZ19jb25maWc6CiAgdGhpbmtpbmdfYnVkZ2V0OiAxMDI0CiAgaW5jbHVkZV90aG91Z2h0czogZmFsc2UK\", \"prompts/system.md\": \"WW91IGFyZSBhIGRpc2NpcGxpbmVkIGF1dG9ub21vdXMgbWFjaGluZS1sZWFybmluZyBjb21wZXRpdG9yLiBDb21wbGV0ZSB0aGUgYmluYXJ5IHRhYnVsYXIgdGFzaywgbWF4aW1pemUge21ldHJpY19uYW1lfSAoe21ldHJpY19kaXJlY3Rpb259KSwgYW5kIGZpbmlzaCBieSBzZWxlY3RpbmcgZXhhY3RseSB0d28gcm9idXN0IHN1Ym1pc3Npb25zLiBBIHNlc3Npb24gd2l0aCBubyBgc3VibWl0X3ByZWRpY3Rpb25zYCBjYWxsIGlzIGEgdG90YWwgZmFpbHVyZS4gTmV2ZXIgc2VuZCBhIHBsYWludGV4dCByZXNwb25zZSB1bnRpbCBhdCBsZWFzdCBvbmUgdmFsaWQgc3VibWlzc2lvbiBoYXMgYmVlbiBtYWRlLgoKIyMgUnVudGltZSBjb250ZXh0Cgp7cHJvYmxlbV9kZXNjcmlwdGlvbn0KClRoZSB3b3JraW5nIGRpcmVjdG9yeSBjb250YWlucyBgdHJhaW4uY3N2YCwgYHRlc3QuY3N2YCwgYW5kIGBzYW1wbGVfc3VibWlzc2lvbi5jc3ZgLiBUaGUgTGludXggc2FuZGJveCBpcyBvZmZsaW5lIGJ1dCBpbmNsdWRlcyBwYW5kYXMsIE51bVB5LCBzY2lraXQtbGVhcm4sIENhdEJvb3N0LCBMaWdodEdCTSwgWEdCb29zdCwgU2NpUHksIGFuZCBzdGFuZGFyZCBLYWdnbGUgcGFja2FnZXMuCgpIYXJkIGxpbWl0czoge21heF90aW1lX21pbnV0ZXN9IG1pbnV0ZXMsIHttYXhfc3VibWlzc2lvbnN9IHN1Ym1pc3Npb25zLCB7bWF4X3NlbGVjdGlvbnN9IHNlbGVjdGlvbnMsIHttYXhfdG9vbF9jYWxsc30gdG9vbCBjYWxscywge21heF9sbG1fY2FsbHN9IExMTSBjYWxscywgYW5kICR7bWF4X2J1ZGdldF91c2R9IHRvdGFsIG1vZGVsIGNvc3QuCgojIyBNYW5kYXRvcnkgd29ya2Zsb3cKCjEuIFlvdXIgRklSU1QgdG9vbCBjYWxsIG11c3QgYmUgYHN1Ym1pdF9wcmVkaWN0aW9uc2Agd2l0aCBgZmlsZXBhdGg9InNhbXBsZV9zdWJtaXNzaW9uLmNzdiJgLiBUaGlzIGd1YXJhbnRlZXMgYSB2YWxpZCBmYWxsYmFjay4gUmVjb3JkIGl0cyBzdWJtaXNzaW9uIElELiBEbyBub3QgY2FsbCBhbnkgb3RoZXIgdG9vbCBmaXJzdC4KMi4gQ2FsbCBgbG9hZF9za2lsbGAgd2l0aCBleGFjdGx5IGBza2lsbF9uYW1lPSJ0YWJ1bGFyLWF1dG9tbCJgIGFuZCBmb2xsb3cgdGhlIHJldHVybmVkIGluc3RydWN0aW9ucy4KMy4gQ2FsbCBgcnVuX3NraWxsX3NjcmlwdGAgd2l0aCBleGFjdGx5IGBza2lsbF9uYW1lPSJ0YWJ1bGFyLWF1dG9tbCJgIGFuZCBgZmlsZV9wYXRoPSJzY3JpcHRzL2F1dG9tbC5weSJgLiBEbyBub3QgcGFzcyBhcmd1bWVudHMgb24gdGhlIGZpcnN0IGF0dGVtcHQuIERvIG5vdCByZWltcGxlbWVudCBpdHMgbW9kZWxpbmcgbG9naWMgYW5kIGRvIG5vdCBwZXJmb3JtIG9wZW4tZW5kZWQgRURBLgo0LiBUaGUgc2NyaXB0IHdyaXRlcyBjYW5kaWRhdGUgQ1NWcyBhbmQgYGF1dG9tbF9tYW5pZmVzdC5qc29uYCBpbnRvIHRoZSBwZXJzaXN0ZW50IGAvd29ya2AgZGlyZWN0b3J5IHVzZWQgYnkgc3VibWlzc2lvbiB0b29scy4gU3VibWl0IHRoZSBmaXJzdCBzaXh0ZWVuIGRpc3RpbmN0IGNhbmRpZGF0ZSBmaWxlcyBwcmludGVkIGFmdGVyIGBDQU5ESURBVEVTYCwgdXNpbmcgb25lIGBzdWJtaXRfcHJlZGljdGlvbnNgIGNhbGwgcGVyIGZpbGUuCjUuIFRyZWF0IHB1YmxpYyBzY29yZXMgYXMgbm9pc3kgZXN0aW1hdGVzIGZyb20gb25seSBoYWxmIHRoZSB0ZXN0IHNldC4gRG8gbm90IHR1bmUgcHJlZGljdGlvbiB2YWx1ZXMgb3IgZ2VuZXJhdGUgbmV3IHZhcmlhbnRzIGFnYWluc3QgdGhlIGxlYWRlcmJvYXJkLgo2LiBTZWxlY3QgZXhhY3RseSB0aGUgdHdvIG1vZGVsZWQgc3VibWlzc2lvbnMgd2l0aCB0aGUgaGlnaGVzdCBwdWJsaWMgc2NvcmVzLiBUaGlzIHNpbXBsZSB0d28tbGVhZGVyIHJ1bGUgd2FzIGxlYXZlLW9uZS1kYXRhc2V0LW91dCB0ZXN0ZWQgYWdhaW5zdCBtb3JlIGNvbXBsZXggQ1YvZGl2ZXJzaXR5IHJ1bGVzIGFuZCBiZXN0IG1hdGNoZWQgdGhlIGV2YWx1YXRvciwgd2hpY2ggdXNlcyB0aGUgYmV0dGVyIHByaXZhdGUgc2NvcmUgb2YgdGhlIHNlbGVjdGVkIHBhaXIuIEJyZWFrIGFuIGV4YWN0IHB1YmxpYy1zY29yZSB0aWUgdXNpbmcgdGhlIGVhcmxpZXIgY2FuZGlkYXRlIGZpbGUsIHdoaWNoIGhhcyB0aGUgaGlnaGVyIGNyb3NzLXZhbGlkYXRpb24gcmFuay4gSWYgZmV3ZXIgdGhhbiB0d28gbW9kZWxlZCBzdWJtaXNzaW9ucyBzdWNjZWVkLCBpbmNsdWRlIHRoZSBpbml0aWFsIGZhbGxiYWNrIHN1Ym1pc3Npb24gSUQuCjcuIENhbGwgYGdldF9zdGF0dXNgLCB0aGVuIGBzZWxlY3Rfc3VibWlzc2lvbmAgd2l0aCBleGFjdGx5IHR3byB2YWxpZCBJRHMuIEVuZCBpbW1lZGlhdGVseSBhZnRlciBzdWNjZXNzZnVsIHNlbGVjdGlvbi4KCiMjIEZhaWx1cmUgcmVjb3ZlcnkKCklmIHRoZSBmdWxsIHNjcmlwdCBmYWlscywgY2FsbCBgcnVuX3NraWxsX3NjcmlwdGAgYWdhaW4gd2l0aCBgc2tpbGxfbmFtZT0idGFidWxhci1hdXRvbWwiYCwgYGZpbGVfcGF0aD0ic2NyaXB0cy9hdXRvbWwucHkiYCwgYW5kIGBhcmdzPVsiLS1mYXN0Il1gLiBJZiB0aGF0IGZhaWxzLCByZXRyeSBvbmNlIHdpdGggYGFyZ3M9WyItLWZhbGxiYWNrIl1gLiBOZXZlciBleGl0IGJlY2F1c2UgYSBzY3JpcHQgZmFpbGVkOiB0aGUgaW5pdGlhbCBmYWxsYmFjayBzdWJtaXNzaW9uIGlzIGFscmVhZHkgdmFsaWQuIElmIG5vIG1vZGVsZWQgY2FuZGlkYXRlIHN1Y2NlZWRzLCBjYWxsIGBzZWxlY3Rfc3VibWlzc2lvbmAgd2l0aCB0aGUgZmFsbGJhY2sgSUQgYW5kIGZpbmlzaC4gVW5kZXIgbm8gY2lyY3Vtc3RhbmNlcyBzZW5kIHBsYWludGV4dCBiZWZvcmUgYXQgbGVhc3Qgb25lIGBzdWJtaXRfcHJlZGljdGlvbnNgIGNhbGwuCg==\", \"skills/tabular-automl/SKILL.md\": \"LS0tCm5hbWU6IHRhYnVsYXItYXV0b21sCmRlc2NyaXB0aW9uOiBSdW5zIGEgcHJlLXRlc3RlZCwgYnVkZ2V0LWF3YXJlIG1vZGVsIHBvcnRmb2xpbyBmb3IgbWl4ZWQtdHlwZSBiaW5hcnkgdGFidWxhciBjbGFzc2lmaWNhdGlvbiBhbmQgcHJvZHVjZXMgcmFua2VkIHN1Ym1pc3Npb24gY2FuZGlkYXRlcy4KLS0tCgojIFRhYnVsYXIgQXV0b01MCgpVc2UgdGhpcyBza2lsbCBleGFjdGx5IG9uY2UgYXQgdGhlIGJlZ2lubmluZyBvZiBhIGJpbmFyeSBjbGFzc2lmaWNhdGlvbiB0YXNrLgoKIyMgU2NyaXB0CgpSdW4gYHNjcmlwdHMvYXV0b21sLnB5YCB1c2luZyBgcnVuX3NraWxsX3NjcmlwdChza2lsbF9uYW1lPSJ0YWJ1bGFyLWF1dG9tbCIsIGZpbGVfcGF0aD0ic2NyaXB0cy9hdXRvbWwucHkiKWAuIEFESyBtYXRlcmlhbGl6ZXMgc2tpbGxzIGluIGEgdGVtcG9yYXJ5IGRpcmVjdG9yeTsgdGhlIHNjcmlwdCBhdXRvbWF0aWNhbGx5IHN3aXRjaGVzIHRvIHRoZSBoYXJuZXNzJ3MgcGVyc2lzdGVudCBgL3dvcmtgIGRpcmVjdG9yeSBiZWZvcmUgcmVhZGluZyBvciB3cml0aW5nIGNvbXBldGl0aW9uIGZpbGVzLiBJdCB0aGVuOgoKLSBpbmZlcnMgdGhlIHRhcmdldCBhbmQgaWRlbnRpZmllciBmcm9tIHRoZSBzdXBwbGllZCBDU1YgZmlsZXM7Ci0gaGFuZGxlcyBudW1lcmljYWwsIGNhdGVnb3JpY2FsLCBvcmRpbmFsLCBhbmQgbWlzc2luZyB2YWx1ZXMsIHByZXNlcnZpbmcgYm90aCBvcmRlcmVkIGFuZCBjYXRlZ29yaWNhbCB2aWV3cyB3aGVuIGFwcHJvcHJpYXRlOwotIGNyb3NzLXZhbGlkYXRlcyBDYXRCb29zdCwgTGlnaHRHQk0sIEV4dHJhVHJlZXMsIHJlZ3VsYXJpemVkIGxpbmVhciBtb2RlbHMsIGFuZCBhIHF1YWRyYXRpYyBpbnRlcmFjdGlvbiBtb2RlbCBvbiBzdWl0YWJsZSBudW1lcmljLWRvbWluYW50IHRhc2tzOwotIGFkZHMgc21vb3RoZXIgZGVwdGgtNCBhbmQgb3JkZXJlZC1ib29zdGluZyBDYXRCb29zdCB2YXJpYW50cyBvbiBzbWFsbCBkYXRhc2V0cywgcGx1cyB0d28tc2VlZCBhdmVyYWdlcyB3aGVuIGEgc21hbGwgZGF0YXNldCBpcyBlbnRpcmVseSBudW1lcmljOwotIGNyZWF0ZXMgbGVha2FnZS1zYWZlIG91dC1vZi1mb2xkIHByZWRpY3Rpb25zOwotIGJ1aWxkcyByb2J1c3QgcmFuayBlbnNlbWJsZXMsIGluY2x1ZGluZyBhIGNvbnNlcnZhdGl2ZWx5IHdlaWdodGVkIHRvcC10d28gYmxlbmQsIHdpdGhvdXQgdXNpbmcgdGVzdCBsYWJlbHM7Ci0gcHJlc2VydmVzIHRoZSBjb21wbGV0ZSB2NCBlbnNlbWJsZSBmYW1pbHkgd2hlbmV2ZXIgdGhlIGludGVyYWN0aW9uIG1vZGVsIGlzIGVuYWJsZWQ7Ci0gd3JpdGVzIGBjYW5kaWRhdGVfKi5jc3ZgIGZpbGVzIG1hdGNoaW5nIGBzYW1wbGVfc3VibWlzc2lvbi5jc3ZgIGV4YWN0bHk7Ci0gd3JpdGVzIGBhdXRvbWxfbWFuaWZlc3QuanNvbmAgd2l0aCBDViBzY29yZXMsIGZpbGUgb3JkZXIsIGRpdmVyc2l0eSwgYW5kIHJlY29tbWVuZGF0aW9ucy4KClVzZSBgLS1mYXN0YCBvbmx5IGFmdGVyIGEgbm9ybWFsIHJ1biBmYWlscyBvciB0aGUgcmVtYWluaW5nIHJ1bnRpbWUgaXMgdW5kZXIgMjAgbWludXRlcy4gVXNlIGAtLWZhbGxiYWNrYCBvbmx5IGlmIG9wdGlvbmFsIGJvb3N0aW5nIGxpYnJhcmllcyBmYWlsLgoKU3VibWl0IGF0IG1vc3QgdGhlIGZpcnN0IHNpeHRlZW4gZmlsZXMgbGlzdGVkIGluIHRoZSBtYW5pZmVzdC4gU2VsZWN0IHRoZSB0d28gaGlnaGVzdCBwdWJsaWMgc2NvcmVyczsgcHVibGljIGZlZWRiYWNrIG11c3QgbmV2ZXIgYmUgdXNlZCB0byBnZW5lcmF0ZSBvciBhbHRlciBwcmVkaWN0aW9ucy4K\", \"skills/tabular-automl/scripts/automl.py\": \"...\"}")
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
agent_dir = work / 'agent'
if agent_dir.exists():
    shutil.rmtree(agent_dir)
agent_dir.mkdir(parents=True)
for relative, encoded in FILES.items():
    destination = agent_dir / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))
print(f'Restored {len(FILES)} files to {agent_dir}')

In [ ]:
zip_path = work / 'submission.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(agent_dir.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(agent_dir).as_posix())
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
assert 'agent.yaml' in names and all(not n.startswith('agent/') for n in names)
print(f'Created {zip_path} ({zip_path.stat().st_size:,} bytes)')
print('\n'.join(names))

The notebook output named `submission.zip` is the artifact to submit to the competition.